# Chapter 8. Data Wrangling: Join, Combine, and Reshape

In [1]:
import pandas as pd
import numpy as np

## 8.3 Reshaping and Pivoting

### Reshaping with Hierarchical Indexing
> - ***stack*** : This **"rotates"** or pivots from the columns in the data to the rows
> - ***unstack*** : This pivots from the rows into the columns
> - By default **the innermost level is unstacked(same with stack)**. You can unstack a different level by passing a level number or name:

In [3]:
data = pd.DataFrame(np.arange(6).reshape((2,3)),
                    index=pd.Index(['Ohio','Colorado'],name='state'),
                    columns=pd.Index(['one','two','three'], name='number'))
data

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [4]:
result = data.stack()
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int64

In [11]:
result.unstack() # = result.unstack(1) = result.unstack('number')

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [15]:
result.unstack(0) # = result.unstack('state')

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


> - Unstacking might introduce missing data if all of the values in the level aren't found in each of the subgroups:

In [18]:
s1 = pd.Series([0,1,2,3], index=['a','b','c','d'])

In [19]:
s2 = pd.Series([4,5,6], index=['c','d','e'])

In [24]:
data2 = pd.concat([s1,s2], keys=['one','two'])
data2

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: int64

In [27]:
data2.unstack()

,a,b,c,d,e
one,0.0,1.0,2.0,3.0,NaN
two,NaN,NaN,4.0,5.0,6.0


In [30]:
data2.unstack().stack(dropna=False)

one  a    0.0
     b    1.0
     c    2.0
     d    3.0
     e    NaN
two  a    NaN
     b    NaN
     c    4.0
     d    5.0
     e    6.0
dtype: float64

- Stacking filters out missing data by default

> - When you unstack in a DataFrame, the level unstacked becomes the lowest level in the result:
> - When calling stack, **we can indicate the name of the axis to stack:**

In [32]:
df = pd.DataFrame({'left':result, 'right':result+5},
                  columns=pd.Index(['left','right'], name='side'))
df

side             left  right
state    number             
Ohio     one        0      5
         two        1      6
         three      2      7
Colorado one        3      8
         two        4      9
         three      5     10

In [33]:
df.unstack('state') # = df.unstack(0)

side   left          right         
state  Ohio Colorado  Ohio Colorado
number                             
one       0        3     5        8
two       1        4     6        9
three     2        5     7       10

In [34]:
df.unstack('state').stack('side')

state         Colorado  Ohio
number side                 
one    left          3     0
       right         8     5
two    left          4     1
       right         9     6
three  left          5     2
       right        10     7

### Pivoting "Long" to "Wide" Format

In [2]:
data = pd.read_csv('Examples/macrodata.csv')
data.head()

,year,quarter,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint
0,1959.0,1.0,2710.349,1707.4,286.898,470.045,1886.9,28.98,139.7,2.82,5.8,177.146,0.00,0.00
1,1959.0,2.0,2778.801,1733.7,310.859,481.301,1919.7,29.15,141.7,3.08,5.1,177.830,2.34,0.74
2,1959.0,3.0,2775.488,1751.8,289.226,491.260,1916.4,29.35,140.5,3.82,5.3,178.657,2.74,1.09
3,1959.0,4.0,2785.204,1753.7,299.356,484.052,1931.3,29.37,140.0,4.33,5.6,179.386,0.27,4.06
4,1960.0,1.0,2847.699,1770.5,331.722,462.199,1955.5,29.54,139.6,3.50,5.2,180.007,2.31,1.19


In [3]:
periods = pd.PeriodIndex(year=data.year, quarter=data.quarter, name='date')
periods

PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

In [4]:
columns = pd.Index(['realgdp','infl','unemp'], name='item') # 일부 열만 선택 1
columns

Index(['realgdp', 'infl', 'unemp'], dtype='object', name='item')

In [5]:
data = data.reindex(columns=columns) # 일부 열만 선택 2
data.head()

item,realgdp,infl,unemp
0,2710.349,0.00,5.8
1,2778.801,2.34,5.1
2,2775.488,2.74,5.3
3,2785.204,0.27,5.6
4,2847.699,2.31,5.2


In [6]:
data.index = periods.to_timestamp('D', 'end')
data.head()

item,realgdp,infl,unemp
date,,,
1959-03-31 23:59:59.999999999,2710.349,0.00,5.8
1959-06-30 23:59:59.999999999,2778.801,2.34,5.1
1959-09-30 23:59:59.999999999,2775.488,2.74,5.3
1959-12-31 23:59:59.999999999,2785.204,0.27,5.6
1960-03-31 23:59:59.999999999,2847.699,2.31,5.2


In [7]:
# method 하나씩 붙여보면서 df의 변화 과정 살펴보면 좋음
ldata = data.stack().reset_index().rename(columns={0:'value'}) 
ldata.head()

,date,item,value
0,1959-03-31 23:59:59.999999999,realgdp,2710.349
1,1959-03-31 23:59:59.999999999,infl,0.000
2,1959-03-31 23:59:59.999999999,unemp,5.800
3,1959-06-30 23:59:59.999999999,realgdp,2778.801
4,1959-06-30 23:59:59.999999999,infl,2.340


> - You might prefer to have a DataFrame containing one column per distinct **item** value indexed by timestamps in the **date** column. DataFrame's ***pivot*** method performs exactly this transformation:
> - The first two values passed are the columns to be used respectively **as the row and column index**, then finally an optional value column to fill the DataFrame.

In [17]:
pivoted = ldata.pivot(index='date', columns='item', values='value')
pivoted

item,infl,realgdp,unemp
date,,,
1959-03-31 23:59:59.999999999,0.00,2710.349,5.8
1959-06-30 23:59:59.999999999,2.34,2778.801,5.1
1959-09-30 23:59:59.999999999,2.74,2775.488,5.3
1959-12-31 23:59:59.999999999,0.27,2785.204,5.6
1960-03-31 23:59:59.999999999,2.31,2847.699,5.2
...,...,...,...
2008-09-30 23:59:59.999999999,-3.16,13324.600,6.0
2008-12-31 23:59:59.999999999,-8.79,13141.920,6.9
2009-03-31 23:59:59.999999999,0.94,12925.410,8.1


In [9]:
ldata['value2'] = np.random.rand(len(ldata))
ldata

,date,item,value,value2
0,1959-03-31 23:59:59.999999999,realgdp,2710.349,0.463709
1,1959-03-31 23:59:59.999999999,infl,0.000,0.018751
2,1959-03-31 23:59:59.999999999,unemp,5.800,0.425709
3,1959-06-30 23:59:59.999999999,realgdp,2778.801,0.449740
4,1959-06-30 23:59:59.999999999,infl,2.340,0.872385
...,...,...,...,...
604,2009-06-30 23:59:59.999999999,infl,3.370,0.848135
605,2009-06-30 23:59:59.999999999,unemp,9.200,0.334126
606,2009-09-30 23:59:59.999999999,realgdp,12990.341,0.946929
607,2009-09-30 23:59:59.999999999,infl,3.560,0.421628


> - By **omitting the last argument**, you obtain a DataFrame with hierarchical columns:

In [10]:
pivoted = ldata.pivot('date','item')
pivoted

value                     value2            \
item                           infl    realgdp unemp      infl   realgdp   
date                                                                       
1959-03-31 23:59:59.999999999  0.00   2710.349   5.8  0.018751  0.463709   
1959-06-30 23:59:59.999999999  2.34   2778.801   5.1  0.872385  0.449740   
1959-09-30 23:59:59.999999999  2.74   2775.488   5.3  0.827124  0.347414   
1959-12-31 23:59:59.999999999  0.27   2785.204   5.6  0.703731  0.299896   
1960-03-31 23:59:59.999999999  2.31   2847.699   5.2  0.786134  0.869682   
...                             ...        ...   ...       ...       ...   
2008-09-30 23:59:59.999999999 -3.16  13324.600   6.0  0.890485  0.171632   
2008-12-31 23:59:59.999999999 -8.79  13141.920   6.9  0.067172  0.976196   
2009-03-31 23:59:59.999999999  0.94  12925.410   8.1  0.387521  0.949878   
2009-06-30 23:59:59.999999999  3.37  12901.504   9.2  0.848135  0.440645   
2009-09-30 23:59:59.999999999  3.56  12990.341   9.6  0.421628  0.946929   

                                         
item                              unemp  
date                                     
1959-03-31 23:59:59.999999999  0.425709  
1959-06-30 23:59:59.999999999  0.175661  
1959-09-30 23:59:59.999999999  0.230586  
1959-12-31 23:59:59.999999999  0.438624  
1960-03-31 23:59:59.999999999  0.855668  
...                                 ...  
2008-09-30 23:59:59.999999999  0.618978  
2008-12-31 23:59:59.999999999  0.214965  
2009-03-31 23:59:59.999999999  0.736503  
2009-06-30 23:59:59.999999999  0.334126  
2009-09-30 23:59:59.999999999  0.248514  

[203 rows x 6 columns]

> - Note that ***pivot*** is equivalent to creating a hierarchical index using ***set_index*** followed by a call to ***unstack***:

In [18]:
# method 하나씩 붙여보면서 df의 변화 과정 살펴보면 좋음
unstacked = ldata.set_index(['date','item']).unstack(level='item')
unstacked

value                     value2            \
item                           infl    realgdp unemp      infl   realgdp   
date                                                                       
1959-03-31 23:59:59.999999999  0.00   2710.349   5.8  0.018751  0.463709   
1959-06-30 23:59:59.999999999  2.34   2778.801   5.1  0.872385  0.449740   
1959-09-30 23:59:59.999999999  2.74   2775.488   5.3  0.827124  0.347414   
1959-12-31 23:59:59.999999999  0.27   2785.204   5.6  0.703731  0.299896   
1960-03-31 23:59:59.999999999  2.31   2847.699   5.2  0.786134  0.869682   
...                             ...        ...   ...       ...       ...   
2008-09-30 23:59:59.999999999 -3.16  13324.600   6.0  0.890485  0.171632   
2008-12-31 23:59:59.999999999 -8.79  13141.920   6.9  0.067172  0.976196   
2009-03-31 23:59:59.999999999  0.94  12925.410   8.1  0.387521  0.949878   
2009-06-30 23:59:59.999999999  3.37  12901.504   9.2  0.848135  0.440645   
2009-09-30 23:59:59.999999999  3.56  12990.341   9.6  0.421628  0.946929   

                                         
item                              unemp  
date                                     
1959-03-31 23:59:59.999999999  0.425709  
1959-06-30 23:59:59.999999999  0.175661  
1959-09-30 23:59:59.999999999  0.230586  
1959-12-31 23:59:59.999999999  0.438624  
1960-03-31 23:59:59.999999999  0.855668  
...                                 ...  
2008-09-30 23:59:59.999999999  0.618978  
2008-12-31 23:59:59.999999999  0.214965  
2009-03-31 23:59:59.999999999  0.736503  
2009-06-30 23:59:59.999999999  0.334126  
2009-09-30 23:59:59.999999999  0.248514  

[203 rows x 6 columns]

### Pivoting "Wide" to "Long" Format
> - **An inverse operation to *pivot*** for DataFrame is ***pandas.melt***.
> - Rather than transforming one column into many in a new DataFrame, it merges multiple columns into one, producing a DataFrame that is longer than the input.

In [19]:
df = pd.DataFrame({'key':['foo','bar','baz'], 'A':[1,2,3],
                   'B':[4,5,6], 'C':[7,8,9]})
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


> - When using ***pandas.melt***, we must indicate which columns (if any) are group indicators.

In [20]:
melted = pd.melt(df, id_vars=['key'])
melted

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6
6,foo,C,7
7,bar,C,8
8,baz,C,9


> - Using pivot, we can reshape back to the ***original layout***:

In [21]:
reshaped = melted.pivot(index='key',columns='variable',values='value')
reshaped

variable,A,B,C
key,,,
bar,2,5,8
baz,3,6,9
foo,1,4,7


In [22]:
reshaped.reset_index()

variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7


> - You can also specify a subset of columns to use as value columns:

In [25]:
pd.melt(df, id_vars=['key'], value_vars=['A','C'])

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,C,7
4,bar,C,8
5,baz,C,9


> - *pandas.melt* can be used **without any group identifiers,** too:

In [26]:
pd.melt(df, value_vars=['A','B','C'])

,variable,value
0,A,1
1,A,2
2,A,3
3,B,4
4,B,5
5,B,6
6,C,7
7,C,8
8,C,9


In [27]:
pd.melt(df, value_vars=['key','A','B'])

,variable,value
0,key,foo
1,key,bar
2,key,baz
3,A,1
4,A,2
5,A,3
6,B,4
7,B,5
8,B,6
